# Gaussian linear regression: unbiased LOO by stratified path sampling

For each replicate, draw an index $i$ uniformly in $\{0,\ldots,n-1\}$ and estimate

$$
\log p(y_i \mid y_{-i}) = \int_0^1 \mathbb{E}_{\pi_{i,\lambda}}[\log p(y_i \mid \theta)]\,d\lambda,
$$

where $\pi_{i,0}(\theta)=p(\theta \mid y_{-i})$ and $\pi_{i,1}(\theta)=p(\theta \mid y)$. Multiplying by $n$ gives an unbiased estimator of the total LOO ELPD.


In [ ]:
from src import *

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.integrate import quad
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper")
np.random.seed(2026)


## Simulated Gaussian linear model


In [ ]:
n, d = 40, 2
true_theta = np.array([2.5, -1.2])
X_data = np.random.randn(n, d)
sigma2_noise = 0.4**2
y_data = X_data @ true_theta + np.random.normal(0, np.sqrt(sigma2_noise), size=n)

prior_mean = np.zeros(d)
prior_cov = np.eye(d) * 10.0
prior_precision = np.linalg.inv(prior_cov)

blr_path = BayesianLinearRegressionTempering(
    X_data,
    y_data,
    prior_mean,
    prior_cov,
    sigma2_noise,
)

print(f"n={n}, d={d}, sigma={np.sqrt(sigma2_noise):.3f}")
print(f"true theta = {true_theta}")


## Path functions using `BayesianLinearRegressionTempering`

The path is the geometric path between the leave-one-out posterior and the full posterior:

$$
\pi_{i,\lambda}(\theta) \propto p(\theta\mid y_{-i})^{1-\lambda}p(\theta\mid y)^\lambda.
$$

Equivalently, up to normalizing constants,

$$
\pi_{i,\lambda}(\theta) \propto p(\theta)\prod_{j\ne i}p(y_j\mid\theta)\,p(y_i\mid\theta)^\lambda.
$$


In [ ]:
# Adaptation of the normal experiment targets to Gaussian LOO.
# For a fixed random index i:
#   log_target0(theta) = log p(theta | y_-i) up to a constant
#   log_target1(theta) = log p(theta | y) up to a constant
#   log_target_path(theta, lambda) = (1-lambda) log_target0 + lambda log_target1
#   grad_log_target_path(theta) = d/dlambda log_target_path(theta, lambda)

loo_index = np.random.randint(n)


def set_loo_index(index_i=None):
    """Choose the leave-one-out index used by the path functions."""
    global loo_index
    if index_i is None:
        index_i = np.random.randint(n)
    loo_index = int(index_i)
    return loo_index


def log_target0(theta):
    """Unnormalized log posterior without observation loo_index."""
    return blr_path.log_posterior_minus_i(theta, loo_index)


def log_target1(theta):
    """Unnormalized full log posterior."""
    return blr_path.log_posterior_full(theta)


def log_target_path(theta, path):
    """Geometric path between p(theta | y_-i) and p(theta | y)."""
    return blr_path.log_path(theta, path, loo_index)


def grad_log_target_path(theta):
    """Derivative in lambda of log_target_path(theta, lambda)."""
    return log_target1(theta) - log_target0(theta)


print(f"Current leave-one-out index: {loo_index}")
print("Endpoint check at prior mean:")
print(f"  path lambda=0: {log_target_path(prior_mean, 0.0):.4f} / target0: {log_target0(prior_mean):.4f}")
print(f"  path lambda=1: {log_target_path(prior_mean, 1.0):.4f} / target1: {log_target1(prior_mean):.4f}")


## Conjugate reference for the LOO score

For Gaussian linear regression the path remains Gaussian. This block computes an analytical LOO reference and checks that the path integral identity is correct.


In [ ]:
def posterior_path_moments(index_i, lambda_val):
    """Mean and covariance of pi_{i,lambda}."""
    x_i = X_data[index_i]
    y_i = y_data[index_i]
    X_minus_i = np.delete(X_data, index_i, axis=0)
    y_minus_i = np.delete(y_data, index_i, axis=0)

    precision = (
        prior_precision
        + X_minus_i.T @ X_minus_i / sigma2_noise
        + lambda_val * np.outer(x_i, x_i) / sigma2_noise
    )
    covariance = np.linalg.inv(precision)
    natural_mean = (
        prior_precision @ prior_mean
        + X_minus_i.T @ y_minus_i / sigma2_noise
        + lambda_val * x_i * y_i / sigma2_noise
    )
    mean = covariance @ natural_mean
    return mean, covariance


def expected_loglik_i(lambda_val, index_i):
    """E_{pi_{i,lambda}}[log p(y_i | theta)]."""
    mean, covariance = posterior_path_moments(index_i, lambda_val)
    x_i = X_data[index_i]
    y_i = y_data[index_i]
    pred_mean = x_i @ mean
    pred_var = x_i @ covariance @ x_i
    expected_sq_error = (y_i - pred_mean) ** 2 + pred_var
    return -0.5 * (np.log(2 * np.pi * sigma2_noise) + expected_sq_error / sigma2_noise)


def path_integral_quad(index_i):
    value, _ = quad(
        lambda lam: expected_loglik_i(lam, index_i),
        0.0,
        1.0,
        epsabs=1e-10,
        epsrel=1e-10,
        limit=100,
    )
    return value


def loo_log_pred_analytic(index_i):
    """Analytical LOO log predictive density p(y_i | y_{-i})."""
    mean_0, covariance_0 = posterior_path_moments(index_i, 0.0)
    x_i = X_data[index_i]
    pred_mean = x_i @ mean_0
    pred_var = sigma2_noise + x_i @ covariance_0 @ x_i
    return stats.norm.logpdf(y_data[index_i], loc=pred_mean, scale=np.sqrt(pred_var))


pointwise_loo_true = np.array([loo_log_pred_analytic(i) for i in range(n)])
pointwise_path_quad = np.array([path_integral_quad(i) for i in range(n)])
true_elpd = np.sum(pointwise_loo_true)
true_value = true_elpd

print(f"Analytical LOO ELPD: {true_elpd:.6f}")
print(f"Max |path integral - analytical LOO|: {np.max(np.abs(pointwise_path_quad - pointwise_loo_true)):.2e}")
print("First 5 pointwise LOO log predictive densities:")
print(np.round(pointwise_loo_true[:5], 4))


## Random-index unbiased estimator with stratified path sampling

Each estimator below draws $i\sim\mathrm{Unif}\{0,\ldots,n-1\}$, estimates the path integral for this index, and returns $n$ times the result.


In [ ]:
def mc_path_integral(index_i, M_lambda):
    lambdas = np.random.uniform(0.0, 1.0, size=M_lambda)
    return np.mean([expected_loglik_i(lam, index_i) for lam in lambdas])


def stratified_path_integral(index_i, L=20, n_per_bin=1):
    grid = np.linspace(0.0, 1.0, L + 1)
    estimator = lambda lam: expected_loglik_i(lam, index_i)
    return stratified_estimator(grid, estimator, n_per_bin=n_per_bin)


def random_index_mc_estimator(M_lambda=20):
    index_i = np.random.randint(n)
    return n * mc_path_integral(index_i, M_lambda=M_lambda)


def random_index_stratified_estimator(L=20, n_per_bin=1):
    index_i = np.random.randint(n)
    return n * stratified_path_integral(index_i, L=L, n_per_bin=n_per_bin)


def full_sum_stratified_estimator(L=20, n_per_bin=1):
    return np.sum([stratified_path_integral(i, L=L, n_per_bin=n_per_bin) for i in range(n)])


## Comparison: uniform Monte Carlo vs stratified path sampling

Both random-index estimators below use the same number of evaluations of $\mathbb{E}_{\pi_{i,\lambda}}[\log p(y_i\mid\theta)]$ per replicate.


In [ ]:
L = 20
n_per_bin = 1
M_lambda = L * n_per_bin
nrep = 1000

mc_estimates = np.array([random_index_mc_estimator(M_lambda=M_lambda) for _ in range(nrep)])
stratified_estimates = np.array([
    random_index_stratified_estimator(L=L, n_per_bin=n_per_bin)
    for _ in range(nrep)
])

results = pd.DataFrame({
    "estimator": ["MC lambda"] * nrep + ["Stratified path"] * nrep,
    "estimate": np.concatenate([mc_estimates, stratified_estimates]),
})

summary = (
    results
    .groupby("estimator")["estimate"]
    .agg(mean="mean", std="std")
    .assign(bias=lambda df: df["mean"] - true_elpd)
)
summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), dpi=150)

sns.boxplot(data=results, x="estimator", y="estimate", ax=axes[0], color="#DCEAF4")
axes[0].axhline(true_elpd, color="#D55E00", linestyle="--", linewidth=2, label="Analytical LOO")
axes[0].set_title("Random-index unbiased LOO estimators")
axes[0].set_xlabel("")
axes[0].set_ylabel("Total ELPD estimate")
axes[0].legend(frameon=False)

sns.histplot(data=results, x="estimate", hue="estimator", bins=35, kde=True, ax=axes[1], element="step")
axes[1].axvline(true_elpd, color="#D55E00", linestyle="--", linewidth=2)
axes[1].set_title("Sampling distribution")
axes[1].set_xlabel("Total ELPD estimate")

plt.tight_layout()
plt.show()


## Optional: stratify the path for every index

This is not the random-index estimator, but it is useful to see the effect of stratifying only the $\lambda$ integration noise while summing over all leave-one-out indices.


In [ ]:
nrep_full = 200
full_stratified_estimates = np.array([
    full_sum_stratified_estimator(L=L, n_per_bin=n_per_bin)
    for _ in range(nrep_full)
])

print(f"Analytical LOO ELPD: {true_elpd:.6f}")
print(f"Full-sum stratified mean: {np.mean(full_stratified_estimates):.6f}")
print(f"Full-sum stratified std: {np.std(full_stratified_estimates, ddof=1):.6f}")
print(f"Bias estimate: {np.mean(full_stratified_estimates) - true_elpd:.6f}")


## Zanella defensive mixture estimator from `gaussian_LOO.py`

The following block keeps the robust mixture-LOO method from the Silva and Zanella paper as a separate benchmark.


In [ ]:
mh_sampler = MixtureMetropolisHastings(
    X_data,
    y_data,
    prior_mean,
    prior_cov,
    sigma2_noise,
)

print("Sampling from q_mix using Metropolis-Hastings...")
q_mix_draws = mh_sampler.sample(num_samples=8000, proposal_std=0.08)
clean_samples = q_mix_draws[2000:]

zanella_estimator = ZanellaLOOEstimator(X_data, y_data, clean_samples, sigma2_noise)
pointwise_loo_zanella, total_elpd_zanella = zanella_estimator.estimate_elpd()

print("\n" + "=" * 54)
print("   Zanella Mixture-LOO Output via MH Sampling")
print("=" * 54)
print(f"Analytical LOO ELPD: {true_elpd:.4f}")
print(f"Zanella mixture estimate: {total_elpd_zanella:.4f}")
print(f"Absolute error: {abs(total_elpd_zanella - true_elpd):.4f}")
print(f"Average log predictive score: {np.mean(pointwise_loo_zanella):.4f}")
print("\nFirst 3 pointwise scores:")
for i in range(3):
    print(
        f"  Point {i}: Zanella={pointwise_loo_zanella[i]:.4f}, "
        f"Analytical={pointwise_loo_true[i]:.4f}"
    )


## MSE comparison for random-index LOO

For each Monte Carlo replication, draw `i` uniformly in `{0, ..., n-1}`, estimate the path integral for that index, and multiply by `n` to target the total LOO ELPD.


In [ ]:
def make_loo_path_estimator(index_i, lambda_grid, k_grid, m_grid, sigmaq, lag):
    """
    Build the UMCMC path-sampling estimator for one leave-one-out index.
    """
    index_i = int(index_i)

    def log_target_path_i(theta, path):
        return blr_path.log_path(theta, path, index_i)

    def grad_log_target_path_i(theta):
        return (
            blr_path.log_posterior_full(theta)
            - blr_path.log_posterior_minus_i(theta, index_i)
        )

    def estimator_i(lam):
        return uestimator_given_lambda(
            lam,
            lambda_grid,
            k_grid,
            m_grid,
            sigmaq,
            lag,
            log_target_path_i,
            grad_log_target_path_i,
        )

    return estimator_i


def compare_mse_loo_budget_grid(
    M_grid,
    L,
    lambda_grid,
    k_grid,
    m_grid,
    sigmaq,
    lag,
    sqrt_m2,
    true_value,
    nrep=10,
    n_samples_per_bin=10,
    plot=True,
):
    """
    Compare MSE for the total LOO ELPD estimator using:
      - q_unbiased_estimator (IS in lambda),
      - optimal_startified_estimator (stratified path sampling).

    At each replication, an index i is sampled uniformly in {0, ..., n-1}.
    The path estimator targets log p(y_i | y_-i), so we multiply by n to target
    the total LOO ELPD.
    """
    rows = []

    for M in M_grid:
        estimates_q = np.zeros(nrep)
        estimates_opt = np.zeros(nrep)

        for r in range(nrep):
            index_i = np.random.randint(n)

            estimator_i = make_loo_path_estimator(
                index_i=index_i,
                lambda_grid=lambda_grid,
                k_grid=k_grid,
                m_grid=m_grid,
                sigmaq=sigmaq,
                lag=lag,
            )

            sqrt_m2_i = sqrt_m2(index_i) if callable(sqrt_m2) else sqrt_m2

            budget_i = build_budget(
                L=L,
                M=M,
                estimator=estimator_i,
                n_samples_per_bin=n_samples_per_bin,
            )

            estimates_q[r] = n * q_unbiased_estimator(
                L,
                M,
                estimator_i,
                sqrt_m2_i,
            )

            estimates_opt[r] = n * optimal_startified_estimator(
                L,
                estimator_i,
                budget_i,
            )

        for label, estimates in [
            ("IS", estimates_q),
            ("Stratified sampling", estimates_opt),
        ]:
            rows.append({
                "M": M,
                "estimator": label,
                "mse": np.mean((estimates - true_value) ** 2),
                "mean": np.mean(estimates),
                "variance": np.var(estimates, ddof=1) if nrep > 1 else 0.0,
                "bias": np.mean(estimates) - true_value,
            })

    df_results = pd.DataFrame(rows)

    if plot:
        sns.set_theme(style="whitegrid", context="paper")

        plt.figure(figsize=(7, 4.5), dpi=150)
        sns.lineplot(
            data=df_results,
            x="M",
            y="mse",
            hue="estimator",
            marker="o",
            linewidth=2,
        )
        plt.xscale("log")
        plt.yscale("log")
        plt.xlabel("Budget N")
        plt.ylabel("MSE")
        plt.title("Comparison of LOO MSE")
        plt.grid(True, which="both", alpha=0.4)
        plt.tight_layout()
        plt.show()

    return df_results


## Example call for the MSE comparison

Run this cell after defining `lambda_grid`, `k_grid`, `m_grid`, `sigmaq`, `lag`, and `sqrt_m2` from the UMCMC tuning step.


In [ ]:
# M_grid = np.array([50, 100, 200, 500, 1000])
# df_mse_loo = compare_mse_loo_budget_grid(
#     M_grid=M_grid,
#     L=L,
#     lambda_grid=lambda_grid,
#     k_grid=k_grid,
#     m_grid=m_grid,
#     sigmaq=sigmaq,
#     lag=lag,
#     sqrt_m2=sqrt_m2,
#     true_value=true_value,
#     nrep=10,
#     n_samples_per_bin=10,
# )
# df_mse_loo
